# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structured approach for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. 

### Dataset Source
This dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

_Dataset summary_: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples using mass spectrometry. It includes fields such as accession, description, coverage, peptide counts, molecular weight (MW), calculated parameters such as pI, and normalized abundances.

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
Let's load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Let's review the available record sets (tables), their fields, and key `@id` references used for programmatic access with mlcroissant.

In [ ]:
# List all record sets and their fields with @id
print("Record sets available:")
for record_set in dataset.record_sets:
    print(f"- RecordSet name: {record_set.name}, @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
We'll extract records from a record set into a DataFrame for further analysis. Please refer to the `@id` values (not names) listed above when selecting the record set and field.

We'll use all available record sets.

In [ ]:
# List of all record sets by @id
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = dict()

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Create dataframe if non-empty
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Show available DataFrames
print('Loaded DataFrames:')
for rs_id in dataframes:
    print(f'RecordSet @id: {rs_id} | Columns: {dataframes[rs_id].columns.tolist()}')

# Show a head preview (using the first DataFrame loaded)
if dataframes:
    rs_id = list(dataframes.keys())[0]
    print(f'Preview of record set: {rs_id}')
    display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Pick a numeric field (referenced by its `@id`), apply filtering, normalization, and group-by as examples. Adapt the `numeric_field` and `group_field` variables as needed, based on the actual field ids/columns shown above.

In [ ]:
# Example: pick a record set and numeric field by @id
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")
    # Try to detect a numeric field automatically
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field (first found): {numeric_field}")
    else:
        # Fallback: try to coerce a common numeric field name
        possible_numeric = [c for c in df.columns if 'abundance' in c.lower() or 'count' in c.lower() or 'weight' in c.lower() or 'coverage' in c.lower()]
        if possible_numeric:
            numeric_field = possible_numeric[0]
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            print(f"Using numeric field: {numeric_field}")
        else:
            numeric_field = None
            print("No numeric field found.")

    if numeric_field:
        # Simple threshold filtering
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean value): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() or 1)
        print(f"\nNormalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a group field (categorical/string)
        group_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) and c != numeric_field]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print("Grouped mean values:")
            display(grouped_df.head())
        else:
            print("No obvious grouping field found for this record set.")
    else:
        print('No numeric analysis possible for this record set.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and one group-by breakdown if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # Group field barplot if available
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(12, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we used the Croissant-compliant FAIR^2 dataset and `mlcroissant` to:
- Programmatically explore available record sets and fields via their `@id`.
- Extract and inspect tabular data using Croissant's schema structure.
- Apply basic filtering, normalization, and aggregation based on field IDs (not names).
- Visualize distributions of protein metrics (or other measured fields).

This provides a reproducible starting point for more advanced proteomics or biomedical data analyses.

**See the Croissant schema documentation, and dataset documentation via [https://sen.science/doi/10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for detailed field/column semantics and analysis recommendations.**